<a href="https://colab.research.google.com/github/LIBY70/Data-Analysis/blob/main/ida_week4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab4: pandas를 이용한 누락 데이터 처리와 데이터 집계

본 실습 자료는 『배워서 바로 써먹는 데이터 분석with 파이썬』 (설진욱, 생능북스)를 참고하여 제작되었습니다.

## 1. 누락 데이터 처리

### 누락 데이터 확인

| 항목 | 설명 |
|------|------|
| `isnull()` | 해당 요소의 값이 NaN인 경우 True, 값이 존재하는 경우 False 반환 / 파이썬의 None도 NaN으로 취급 |
| `notnull()` | `isnull()`과 True/False를 반대로 반환 / `myseries[myseries.notnull()]`로 참인 항목만 출력 가능 |

In [2]:
import numpy as np
from pandas import Series, DataFrame

myseries = Series([1, 2, np.nan, 4, 5])
myseries

,0
0,1.0
1,2.0
2,NaN
3,4.0
4,5.0


- `isnull()`: 값이 NaN이면 True 반환 (파이썬의 None도 NaN으로 취급)
- `notnull()`: `isnull()`의 반대
- `myseries[myseries.notnull()]`: 시리즈에서 NaN을 제외한 데이터만 추출

In [3]:
myseries.isnull()

,0
0,False
1,False
2,True
3,False
4,False


In [5]:
myseries.notnull()

,0
0,True
1,True
2,False
3,True
4,True


In [6]:
# NaN을 제외한 데이터만 추출
myseries[myseries.notnull()]

,0
0,1.0
1,2.0
3,4.0
4,5.0


- `dropna()`: 누락된 데이터가 있는 행과 열을 제외시키고, 나머지 행 또는 열을 조회 (기본값: NaN이 하나라도 있는 모든 행 배제)

In [7]:
myseries.dropna()

,0
0,1.0
1,2.0
3,4.0
4,5.0


`excel02.csv`: 일부 데이터에 NaN이 포함된 샘플 파일

In [21]:
import pandas as pd

myframe = pd.read_csv('excel02.csv', index_col='이름')
myframe

,국어,영어,수학
이름,,,
강감찬,90.0,85.0,92.0
홍길동,82.0,NaN,NaN
박영희,NaN,NaN,NaN
김철수,NaN,70.0,88.0


- `dropna()` 기본값: NaN이 하나라도 있는 행 배제 → '강감찬'(NaN 없음)만 조회

In [9]:
myframe.dropna()

,국어,영어,수학
이름,,,
강감찬,90.0,85.0,92.0


- `how='all'`: 모든 값이 NaN인 행만 배제 → '박영희'(전 과목 미응시)만 제외

In [11]:
myframe.dropna(how='all')

,국어,영어,수학
이름,,,
강감찬,90.0,85.0,92.0
홍길동,82.0,NaN,NaN
김철수,NaN,70.0,88.0


- `subset`: 특정 열에 대해서만 NaN 여부 체크 → '영어' 컬럼에 NaN이 있는 행만 삭제

In [15]:
myframe.dropna(subset=['영어'])

,국어,영어,수학
이름,,,
강감찬,NaN,85.0,92.0
김철수,NaN,70.0,88.0


- `axis=1`: 열 방향으로 적용 / `how='all'`: 모든 값이 NaN인 열만 삭제 → 해당 열 없으므로 제거 없음

In [13]:
myframe.dropna(axis=1, how='all')

,국어,영어,수학
이름,,,
강감찬,90.0,85.0,92.0
홍길동,82.0,NaN,NaN
박영희,NaN,NaN,NaN
김철수,NaN,70.0,88.0


- '강감찬'과 '홍길동'의 '국어' 점수를 NaN으로 변경 → '국어' 열 전체가 NaN

In [17]:
myframe.loc[['강감찬', '홍길동'], '국어'] = np.nan
myframe

,국어,영어,수학
이름,,,
강감찬,NaN,85.0,92.0
홍길동,NaN,NaN,NaN
박영희,NaN,NaN,NaN
김철수,NaN,70.0,88.0


- `axis=1`, `how='all'`: '국어' 열 전체가 NaN이므로 '국어' 열 삭제

In [18]:
myframe.dropna(axis=1, how='all')

,영어,수학
이름,,
강감찬,85.0,92.0
홍길동,NaN,NaN
박영희,NaN,NaN
김철수,70.0,88.0


- `thresh=2`: NaN이 아닌 데이터가 2개 이상인 행만 조회

In [27]:
myframe.dropna(thresh=2)

,국어,영어,수학
이름,,,
강감찬,90.0,85.0,92.0
김철수,NaN,70.0,88.0


- `axis=1` 기본값(`how='any'`): NaN이 하나라도 있는 열 모두 제거 → 빈 DataFrame 반환

In [25]:
myframe.dropna(axis=1)

""
이름
강감찬
홍길동
박영희
김철수


### 누락된 데이터 값 채우기

`fillna()`: 누락된 데이터를 다른 값으로 채워 넣는 함수

In [26]:
import pandas as pd
import numpy as np

myframe = pd.read_csv('excel02.csv', index_col='이름')
myframe

,국어,영어,수학
이름,,,
강감찬,90.0,85.0,92.0
홍길동,82.0,NaN,NaN
박영희,NaN,NaN,NaN
김철수,NaN,70.0,88.0


- `inplace=False` (기본값): 원본을 변경하지 않고 결과만 반환

In [28]:
myframe.fillna(0, inplace=False)

,국어,영어,수학
이름,,,
강감찬,90.0,85.0,92.0
홍길동,82.0,0.0,0.0
박영희,0.0,0.0,0.0
김철수,0.0,70.0,88.0


In [29]:
# 원본은 변경되지 않음
myframe

,국어,영어,수학
이름,,,
강감찬,90.0,85.0,92.0
홍길동,82.0,NaN,NaN
박영희,NaN,NaN,NaN
김철수,NaN,70.0,88.0


- `inplace=True`: 복사본을 생성하지 않고 객체 자체를 변경

In [30]:
myframe.fillna(0, inplace=True)
myframe

,국어,영어,수학
이름,,,
강감찬,90.0,85.0,92.0
홍길동,82.0,0.0,0.0
박영희,0.0,0.0,0.0
김철수,0.0,70.0,88.0


In [31]:
myframe = pd.read_csv('excel02.csv', index_col='이름')
myframe

,국어,영어,수학
이름,,,
강감찬,90.0,85.0,92.0
홍길동,82.0,NaN,NaN
박영희,NaN,NaN,NaN
김철수,NaN,70.0,88.0


- 딕셔너리로 컬럼별 다른 값으로 치환 가능 → 국어: 15, 영어: 25, 수학: 35

In [34]:
myframe.fillna({'국어': 15, '영어': 25, '수학': 35})

,국어,영어,수학
이름,,,
강감찬,90.0,85.0,92.0
홍길동,82.0,25.0,35.0
박영희,15.0,25.0,35.0
김철수,15.0,70.0,88.0


In [33]:
myframe = pd.read_csv('excel02.csv', index_col='이름')
myframe

,국어,영어,수학
이름,,,
강감찬,90.0,85.0,92.0
홍길동,82.0,NaN,NaN
박영희,NaN,NaN,NaN
김철수,NaN,70.0,88.0


- NaN을 해당 컬럼의 평균값으로 대체: `mean()`으로 평균 계산 후 딕셔너리 형식으로 치환

In [36]:
fill_values = {
    '국어': myframe['국어'].mean(),
    '영어': myframe['영어'].mean(),
    '수학': myframe['수학'].mean()
}
myframe.fillna(fill_values)

,국어,영어,수학
이름,,,
강감찬,90.0,85.0,92.0
홍길동,82.0,77.5,90.0
박영희,86.0,77.5,90.0
김철수,86.0,70.0,88.0


## 2. 데이터 집계

### 데이터 집계 관련 함수

| 항목 | 설명 |
|------|------|
| `sum()` | 배열 전체 혹은 특정 축에 대한 모든 원소의 합 계산 / 크기가 0인 배열에 대한 연산 결과는 0 |
| `mean()` | 산술 평균 / 크기가 0인 배열에 대한 연산 결과는 NaN |
| `std()`, `var()` | 표준 편차와 분산 / 선택적으로 자유도 설정 가능, 분모의 기본값은 n |
| `min()`, `max()` | 최솟값, 최댓값 |
| `argmin()`, `argmax()` | 최소 원소의 색인 값, 최대 원소의 색인 값 |
| `cumsum()` | 각 원소의 누적 합 (cumulative sum) |
| `cumprod()` | 각 원소의 누적 곱 |
| `describe()` | 시리즈에도 사용 가능 / 기술 통계 정보 반환 |

### 기술 통계 계산과 요약

In [37]:
import numpy as np
from pandas import Series, DataFrame

myframe = DataFrame(
    {
        '국어': [75, np.nan, 90, 95],
        '영어': [np.nan, 70, 85, 80],
        '수학': [83, 88, 92, np.nan]
    },
    index=['이순신', '김유신', '강감찬', '계백']
)
myframe

,국어,영어,수학
이순신,75.0,NaN,83.0
김유신,NaN,70.0,88.0
강감찬,90.0,85.0,92.0
계백,95.0,80.0,NaN


- 집계 함수 기본값: NaN 배제 후 연산
- `sum()`: 각 컬럼의 합을 담은 시리즈 반환

In [38]:
myframe.sum()

,0
국어,260.0
영어,235.0
수학,263.0


- `axis=1`: 행 방향으로 합산 ,행축을 없앤다. , 열의 합을 구한다.

In [39]:
myframe.sum(axis=1)

,0
이순신,158.0
김유신,158.0
강감찬,267.0
계백,175.0


- `skipna=False`: NaN이 하나라도 있으면 연산 결과를 NaN으로 반환 True가 기본

In [40]:
myframe.sum(axis=1, skipna=False)

,0
이순신,NaN
김유신,NaN
강감찬,267.0
계백,NaN


- `skipna=True`: NaN 무시하고 연산 수행

In [42]:
myframe.sum(axis=1, skipna=True)

,0
이순신,158.0
김유신,158.0
강감찬,267.0
계백,175.0


- `max(skipna=False)`: NaN 포함 시 결과도 NaN → '이순신'(NaN 포함)은 NaN 반환

In [43]:
myframe.max(axis=1, skipna=False)

,0
이순신,NaN
김유신,NaN
강감찬,92.0
계백,NaN


- `max(skipna=True)`: NaN 배제 후 최댓값

In [47]:
myframe.max(axis=1, skipna=True)

,0
이순신,83.0
김유신,88.0
강감찬,92.0
계백,95.0


- `idxmax()`: 최댓값을 가진 색인 반환 → 국어 최고점: '계백'

In [45]:
myframe.idxmax()

,0
국어,계백
영어,강감찬
수학,강감찬


누적 관련 함수
- `cumsum()`: 누적 합 (cumulative sums)
- `cumprod()`: 누적 곱 (cumulative products)
- `cummin()`: 누적 최솟값 (cumulative minimal)
- `cummax()`: 누적 최댓값 (cumulative maximal)

In [48]:
myframe

,국어,영어,수학
이순신,75.0,NaN,83.0
김유신,NaN,70.0,88.0
강감찬,90.0,85.0,92.0
계백,95.0,80.0,NaN


In [49]:
myframe.cumsum()

,국어,영어,수학
이순신,75.0,NaN,83.0
김유신,NaN,70.0,171.0
강감찬,165.0,155.0,263.0
계백,260.0,235.0,NaN


In [50]:
myframe.cumprod()

,국어,영어,수학
이순신,75.0,NaN,83.0
김유신,NaN,70.0,7304.0
강감찬,6750.0,5950.0,671968.0
계백,641250.0,476000.0,NaN


In [51]:
myframe.cummin()

,국어,영어,수학
이순신,75.0,NaN,83.0
김유신,NaN,70.0,83.0
강감찬,75.0,70.0,83.0
계백,75.0,70.0,NaN


In [52]:
myframe.cummax()

,국어,영어,수학
이순신,75.0,NaN,83.0
김유신,NaN,70.0,88.0
강감찬,90.0,85.0,92.0
계백,95.0,85.0,NaN


NaN 값 처리 방법
- 기본값 정의 후 대입
- NaN이 아닌 값들의 평균값으로 대체: `mean()`으로 평균 계산 후 딕셔너리 형식으로 치환

In [53]:
fill_values = {
    '국어': myframe['국어'].mean(),
    '영어': myframe['영어'].mean(),
    '수학': myframe['수학'].mean()
}
newframe = myframe.fillna(fill_values)
newframe

,국어,영어,수학
이순신,75.000000,78.333333,83.000000
김유신,86.666667,70.000000,88.000000
강감찬,90.000000,85.000000,92.000000
계백,95.000000,80.000000,87.666667


`describe()`: 한 번에 여러 통계 결과 반환 (시리즈/데이터프레임 모두 사용 가능)
- `count`: 빈도 수
- `mean`: 평균값
- `std`: 표준 편차
- `min` ~ `max`: 4분위수 (전체 데이터를 4등분하여 25, 50, 75, 100%에 해당하는 값)

In [54]:
myframe.describe()

,국어,영어,수학
count,3.000000,3.000000,3.000000
mean,86.666667,78.333333,87.666667
std,10.408330,7.637626,4.509250
min,75.000000,70.000000,83.000000
25%,82.500000,75.000000,85.500000
50%,90.000000,80.000000,88.000000
75%,92.500000,82.500000,90.000000
max,95.000000,85.000000,92.000000


- NaN을 평균값으로 대체한 데이터프레임의 통계

In [55]:
newframe.describe()

,국어,영어,수학
count,4.000000,4.000000,4.000000
mean,86.666667,78.333333,87.666667
std,8.498366,6.236096,3.681787
min,75.000000,70.000000,83.000000
25%,83.750000,76.250000,86.500000
50%,88.333333,79.166667,87.833333
75%,91.250000,81.250000,89.000000
max,95.000000,85.000000,92.000000
